# Providers Exploratory Data Analysis

## Purpose

Assess whether provider records are complete, uniquely identified, consistently classified, and correctly connected to practitioners and downstream screenings.

## Files used

- `Providers.csv` — provider details
- `Practitioners.csv` — practitioners assigned to providers
- `Screenings.csv` — screenings delivered by practitioners

The source contains only two providers, so this notebook validates the current records and relationships but cannot establish broad population-level patterns.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

def find_raw_data_dir(start: Path = Path.cwd()) -> Path:
    for directory in (start, *start.parents):
        candidates = (
            directory / "data" / "raw",
            directory / "data-analytics" / "data" / "raw",
        )
        for candidate in candidates:
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError("Could not locate data-analytics/data/raw")

RAW_DATA_DIR = find_raw_data_dir()

## 1. Load the data

In [2]:
providers = pd.read_csv(RAW_DATA_DIR / "Providers.csv")
practitioners = pd.read_csv(RAW_DATA_DIR / "Practitioners.csv")
screenings = pd.read_csv(RAW_DATA_DIR / "Screenings.csv")

pd.DataFrame({
    "dataset": ["Providers", "Practitioners", "Screenings"],
    "rows": [len(providers), len(practitioners), len(screenings)],
    "columns": [len(providers.columns), len(practitioners.columns), len(screenings.columns)],
})

,dataset,rows,columns
0,Providers,2,6
1,Practitioners,3,8
2,Screenings,144,8


## 2. Inspect provider structure and completeness

In [3]:
providers

,provider_id,name,provider_type,country,verification_status,active
0,PRO-001,Bophelo Preventive Health Services,clinic,Botswana,verified,True
1,PRO-002,ActiveWork Fitness Services,fitness company,Botswana,verified,True


In [4]:
provider_profile = pd.DataFrame({
    "data_type": providers.dtypes.astype(str),
    "missing_count": providers.isna().sum(),
    "unique_values": providers.nunique(dropna=False),
})
provider_profile

,data_type,missing_count,unique_values
provider_id,object,0,2
name,object,0,2
provider_type,object,0,2
country,object,0,1
verification_status,object,0,1
active,bool,0,1


In [5]:
required_columns = [
    "provider_id", "name", "provider_type", "country",
    "verification_status", "active",
]
assert set(required_columns).issubset(providers.columns)
assert providers[required_columns].notna().all().all()
print("All required provider fields are present and complete.")

All required provider fields are present and complete.


## 3. Validate provider identifiers

In [6]:
identifier_quality = pd.Series({
    "missing_provider_ids": providers["provider_id"].isna().sum(),
    "duplicate_provider_ids": providers["provider_id"].duplicated().sum(),
    "invalid_provider_id_formats": (~providers["provider_id"].str.match(r"^PRO-[0-9]{3}$", na=False)).sum(),
})
identifier_quality.to_frame("count")

,count
missing_provider_ids,0
duplicate_provider_ids,0
invalid_provider_id_formats,0


In [17]:
assert providers["provider_id"].notna().all()
assert providers["provider_id"].is_unique
assert providers["provider_id"].str.match(r"^PRO-[0-9]{3}$", na=False).all()
print("Provider identifiers are complete, unique and consistently formatted.")

Provider identifiers are complete, unique and consistently formatted.


## 4. Review provider classifications and status

In [8]:
for column in ["provider_type", "country", "verification_status", "active"]:
    display(providers[column].value_counts(dropna=False).to_frame("provider_count"))

,provider_count
provider_type,
clinic,1
fitness company,1


,provider_count
country,
Botswana,2


,provider_count
verification_status,
verified,2


,provider_count
active,
True,2


In [9]:
text_columns = ["name", "provider_type", "country", "verification_status"]
whitespace_issues = pd.Series({
    column: providers[column].ne(providers[column].str.strip()).sum()
    for column in text_columns
})
whitespace_issues.to_frame("rows_with_outer_whitespace")

,rows_with_outer_whitespace
name,0
provider_type,0
country,0
verification_status,0


## 5. Validate provider-to-practitioner relationships

In [10]:
unknown_provider_ids = sorted(
    set(practitioners["provider_id"]) - set(providers["provider_id"])
)
print("Practitioner records with unknown provider IDs:", len(unknown_provider_ids))
unknown_provider_ids

Practitioner records with unknown provider IDs: 0


[]

In [11]:
provider_practitioner_join = practitioners.merge(
    providers[["provider_id", "name", "provider_type", "verification_status", "active"]].rename(
        columns={
            "name": "provider_name",
            "verification_status": "provider_verification_status",
            "active": "provider_active",
        }
    ),
    on="provider_id",
    how="left",
    validate="many_to_one",
    indicator="provider_join_status",
)
provider_practitioner_join

,practitioner_id,provider_id,practitioner_code,practitioner_type,registration_body,registration_number,verification_status,active,provider_name,provider_type,provider_verification_status,provider_active,provider_join_status
0,PRA-001,PRO-001,P80-PRA-001,nurse,BHPC,BHPC-RN-2814,verified,True,Bophelo Preventive Health Services,clinic,verified,True,both
1,PRA-002,PRO-001,P80-PRA-002,nurse,BHPC,BHPC-RN-3176,verified,True,Bophelo Preventive Health Services,clinic,verified,True,both
2,PRA-003,PRO-002,P80-PRA-003,fitness coach,other,FIT-BW-084,verified,True,ActiveWork Fitness Services,fitness company,verified,True,both


In [12]:
practitioners_per_provider = (
    provider_practitioner_join.groupby(
        ["provider_id", "provider_name"], dropna=False
    )["practitioner_id"]
    .nunique()
    .rename("practitioner_count")
    .reset_index()
)
practitioners_per_provider

,provider_id,provider_name,practitioner_count
0,PRO-001,Bophelo Preventive Health Services,2
1,PRO-002,ActiveWork Fitness Services,1


In [13]:
assert not unknown_provider_ids
assert len(provider_practitioner_join) == len(practitioners)
assert provider_practitioner_join["provider_join_status"].eq("both").all()
print("All practitioners link to a known provider without losing rows.")

All practitioners link to a known provider without losing rows.


## 6. Validate downstream screening coverage

In [14]:
screening_provider_join = screenings.merge(
    practitioners[["practitioner_id", "provider_id"]],
    on="practitioner_id",
    how="left",
    validate="many_to_one",
    indicator="practitioner_join_status",
).merge(
    providers[["provider_id", "name"]].rename(columns={"name": "provider_name"}),
    on="provider_id",
    how="left",
    validate="many_to_one",
    indicator="provider_join_status",
)
screening_provider_join.head()

,screening_id,participation_id,programme_service_id,practitioner_id,screened_at,status,notes,created_at,provider_id,practitioner_join_status,provider_name,provider_join_status
0,SCR-0001,PAR-001,PS-001,PRA-002,2026-06-18T07:09:00Z,completed,NaN,2026-06-18T06:30:00Z,PRO-001,both,Bophelo Preventive Health Services,both
1,SCR-0002,PAR-001,PS-002,PRA-002,2026-06-18T07:13:00Z,completed,NaN,2026-06-18T06:30:00Z,PRO-001,both,Bophelo Preventive Health Services,both
2,SCR-0003,PAR-001,PS-003,PRA-002,2026-06-18T07:17:00Z,completed,NaN,2026-06-18T06:30:00Z,PRO-001,both,Bophelo Preventive Health Services,both
3,SCR-0004,PAR-001,PS-004,PRA-002,2026-06-18T07:21:00Z,completed,NaN,2026-06-18T06:30:00Z,PRO-001,both,Bophelo Preventive Health Services,both
4,SCR-0005,PAR-001,PS-005,PRA-002,2026-06-18T07:25:00Z,completed,NaN,2026-06-18T06:30:00Z,PRO-001,both,Bophelo Preventive Health Services,both


In [15]:
screenings_per_provider = (
    screening_provider_join.groupby(
        ["provider_id", "provider_name"], dropna=False
    )["screening_id"]
    .nunique()
    .rename("screening_count")
    .reset_index()
)
screenings_per_provider

,provider_id,provider_name,screening_count
0,PRO-001,Bophelo Preventive Health Services,120
1,PRO-002,ActiveWork Fitness Services,24


In [16]:
assert len(screening_provider_join) == len(screenings)
assert screening_provider_join["practitioner_join_status"].eq("both").all()
assert screening_provider_join["provider_join_status"].eq("both").all()
print("All screenings link to a known practitioner and provider without losing rows.")

All screenings link to a known practitioner and provider without losing rows.


## 7. Findings and recommendations

The current provider records are complete and use unique, consistently formatted identifiers. Both providers are active and verified, and every practitioner links to a known provider. All screening records also connect successfully to providers through practitioners without row loss.

The dataset is too small to evaluate provider diversity or compare performance reliably: it contains two providers across two provider types. Provider-level screening counts describe current activity but should not be interpreted as quality or performance measures.

Next steps:

- Retain the current provider identifier and relationship rules.
- Define controlled values for provider type and verification status as the catalogue grows.
- Add verification dates, contact or location metadata, and service capabilities when operationally appropriate.
- Repeat distribution and coverage analysis when more providers are available.
- Pair activity counts with outcome, capacity, and service-quality measures before comparing providers.